# LC 130 — Surrounded Regions
**Difficulty:** Medium | **Pattern:** Boundary-seeded DFS/BFS

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Instead of asking "which 'O' cells
are surrounded?", ask the <em>opposite</em>: "which 'O' cells
touch the border?" Those are safe. Mark them, then flip
everything: unsafe 'O'→'X', safe marker→'O'.
</div>

## Official Problem Statement

Given an `m x n` matrix `board` containing `'X'` and `'O'`,
**capture all regions** that are 4-directionally surrounded by
`'X'`.

A region is captured by flipping all `'O'`s into `'X'`s in
that surrounded region. An `'O'` region is **not** captured if
it is on the border or connected to a border `'O'`.

Modify the board **in-place**.

**Constraints:**
- `m == board.length`, `n == board[i].length`
- `1 <= m, n <= 200`
- `board[i][j]` is `'X'` or `'O'`

## What This Is Actually Asking

An 'O' survives only if it has a path to the border through
other 'O' cells. The clever inversion: flood-fill from every
border 'O' to mark the entire safe component with a sentinel
'S'. After the fill, any remaining 'O' is landlocked → flip it
to 'X'. Finally, restore 'S' → 'O'. Three sweeps total:
mark safe, flip unsafe, restore.

## Walk Through an Example by Hand

```
Initial:
  X X X X
  X O O X
  X X O X
  X O X X

Step 1 — mark safe 'O's from border:
  Border 'O': (3,1) → DFS from (3,1)
  (3,1) has no 'O' neighbours → just mark (3,1)='S'
  Result:
  X X X X
  X O O X
  X X O X
  X S X X

Step 2 — flip remaining 'O' to 'X':
  X X X X
  X X X X
  X X X X
  X S X X

Step 3 — restore 'S' to 'O':
  X X X X
  X X X X
  X X X X
  X O X X
```

## The Picture

```
Board:           After safe-mark:    After flip+restore:
X X X X          X X X X             X X X X
X O O X    →     X O O X      →      X X X X
X X O X          X X O X             X X X X
X O X X          X S X X             X O X X

Algorithm:
  Phase 1 — Boundary DFS/BFS:
    for each border cell (r,c):
      if board[r][c]=='O':
          dfs(r,c)  # marks 'O'→'S'

  Phase 2 — Single pass flip:
    for each cell:
      'O' → 'X'   (surrounded, capture it)
      'S' → 'O'   (safe, restore it)
      'X' → 'X'   (unchanged)

DFS marks 4-directional 'O' neighbours recursively.
BFS variant: use deque, same logic.
```

## When To Use This Pattern

- When you need to find cells that **cannot reach the border**,
  think **invert: flood-fill from border, mark safe**.
- When a direct "is it surrounded?" flood-fill is costly,
  think **boundary-seeded DFS/BFS as the starting point**.
- When you must modify a grid in-place using a sentinel value,
  think **three-phase: mark, flip unsafe, restore sentinel**.
- When the grid has up to 200×200 cells, DFS with recursion
  may hit Python stack limits — BFS with a deque is safer.
- When "connected to the edge" = "safe", this pattern applies.

## The Approach

Iterate over the four border edges. For any 'O' found, run a
DFS (or BFS) that marks all connected 'O' cells as 'S'
(safe). Then do a single pass over the entire board: 'O'
becomes 'X', 'S' becomes 'O', 'X' stays 'X'. This correctly
captures all surrounded regions without ever explicitly
checking whether a region is fully enclosed.

In [ ]:
from typing import List
from collections import deque

In [ ]:
def test_harness(func):
    import copy

    cases = [
        (
            [["X","X","X","X"],
             ["X","O","O","X"],
             ["X","X","O","X"],
             ["X","O","X","X"]],
            [["X","X","X","X"],
             ["X","X","X","X"],
             ["X","X","X","X"],
             ["X","O","X","X"]]
        ),
        (
            [["X"]],
            [["X"]]   # single cell
        ),
        (
            [["O"]],
            [["O"]]   # border O, stays
        ),
        (
            [["O","O"],["O","O"]],
            [["O","O"],["O","O"]]  # all border → no flip
        ),
    ]
    passed = 0
    for board, expected in cases:
        b = copy.deepcopy(board)
        func(b)
        status = "PASSED" if b == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"  {status}:")
            print(f"    got     {b}")
            print(f"    expected {expected}")
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def solve(board: List[List[str]]) -> None:
    """
    Capture surrounded 'O' regions by flipping them to 'X'.
    Modifies board in-place.

    Strategy:
      Phase 1: BFS/DFS from every border 'O'; mark safe→'S'.
      Phase 2: Sweep board:
               'O'→'X'  (surrounded)
               'S'→'O'  (safe, restore)

    Args:
        board: m x n grid of 'X' and 'O', modified in-place
    Returns:
        None
    """
    # Debug: print board dimensions
    # m, n = len(board), len(board[0])
    # print(f"Board {m}x{n}")

    pass

    # Debug: print board after phase 1 (S-marks visible)
    # for row in board: print(row)

In [ ]:
# Uncomment and run when solution is ready
# test_harness(solve)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Flood-fill every 'O' inward | O((m·n)²) | O(m·n) | Redundant checks |
| Union-Find with virtual border | O(m·n·α) | O(m·n) | Elegant but verbose |
| **Boundary DFS/BFS + sweep** | **O(m·n)** | **O(m·n)** | Optimal |

## Real World Connection

At **Citi**, compliance systems must identify trading positions
that are completely enclosed by restricted counterparties —
boundary-seeded BFS finds the safe positions first, then
everything else is flagged. In **AWS VPC** security groups,
subnets fully enclosed by deny-all rules (with no path to an
internet gateway) are structurally surrounded regions. For a
**data engineer**, tables in a data lake that have no lineage
path to any external source or sink are "surrounded" dead ends.
The inversion trick — start from the boundary, mark safe,
flip the rest — applies broadly to any reachability problem.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra